# W01-A · Regex on real reviews *(guided lab, ~30 min)*

**Business question:** You manage the *Appliances & Parts* category. 30,000 customer reviews came in.
Without reading all of them, what can you **pull out** of the text?

By the end of this notebook you will be able to:
1. use `re.search`, `re.findall` and `re.sub` in Python
2. apply the same patterns to a whole column with pandas `.str` methods
3. spot **false positives** and **false negatives** in your own patterns

Cells marked ✍️ are for you to complete. Replace `...` with your code, then run the ✅ check cell underneath.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 160)
DATA = "../data/"   # notebooks live in ADS6703/w01/, data in ADS6703/data/

reviews = pd.read_csv(DATA + "reviews.csv.gz")
products = pd.read_csv(DATA + "products.csv.gz")
print(reviews.shape, products.shape)
reviews.head()

## 1 · Meet the data (3 min)

In [ ]:
reviews["rating"].value_counts(normalize=True).sort_index().round(3)

> 💬 **Discuss:** About three quarters of the reviews are 5★. What does that do to an "average rating" on a product page?
> Where would you look to find the real problems?

## 2 · Regex in plain Python (8 min)

Three functions do most of the work:

| function | returns |
|---|---|
| `re.search(p, s)` | the **first** match (or `None`) |
| `re.findall(p, s)` | a **list** of all matches |
| `re.sub(p, repl, s)` | the string with matches **replaced** |

Always write patterns as **raw strings**: `r"\d+"`, not `"\d+"`.

In [ ]:
s = "Fits my Whirlpool WDT970SAHZ0 perfectly. Paid $24.99 here, Sears wanted $80 for the same part!"

print(re.findall(r"\$\d+", s))               # first attempt
print(re.findall(r"\$\d+(?:\.\d{2})?", s))    # allow optional cents

The first pattern **under-matches**: it cuts `$24.99` down to `$24`.
`(?: … )?` is an *optional non-capturing group*: "cents, if present".

⚠️ **Gotcha:** if your pattern contains a *capturing* group `( … )`, `findall` returns **only the group**. Try it:

In [ ]:
print(re.findall(r"\$\d+(\.\d{2})?", s))   # capturing group: surprise!

### ✍️ Exercise 1: model numbers

Appliance parts are identified by model numbers like `WDT970SAHZ0`, `UKF8001` and `LT1000PC`:
**two or more capital letters, then three or more digits, then optionally more capitals/digits**.
Use `\b` so that you don't match inside longer words.

In [ ]:
model_pattern = r"..."   # ✍️ your pattern here

tests = "Replaced UKF8001 with LT1000PC. My GE fridge (GSS25GSHSS) and a K200 brewer. Call 1800 now."
re.findall(model_pattern, tests)

In [ ]:
# ✅ check
assert re.findall(model_pattern, tests) == ["UKF8001", "LT1000PC"], "not quite: look at which ones you missed or over-matched"
print("✅ Nice. Note GSS25GSHSS and K200 were missed. Is that a false negative you can live with?")

## 3 · Scale up with pandas (8 min)

Every `re` function has a pandas twin that works on a whole column:

| pandas | does |
|---|---|
| `.str.contains(p)` | True/False per row, good for **flags** |
| `.str.findall(p)` | list of matches per row |
| `.str.extract(p)` | capturing groups → **new columns** |
| `.str.replace(p, r, regex=True)` | cleaning |

Reviews contain raw HTML line breaks (`<br />`). Let's count and clean them.

In [ ]:
has_br = reviews["text"].str.contains(r"<br\s*/?>")
print(f"{has_br.sum():,} reviews contain <br> tags")

reviews["clean"] = reviews["text"].str.replace(r"<br\s*/?>", " ", regex=True)
reviews.loc[has_br, ["text", "clean"]].head(3)

In [ ]:
# Do reviews that talk about price rate differently?
mentions_price = reviews["clean"].str.contains(r"\$\d")
reviews.groupby(mentions_price)["rating"].agg(["count", "mean"]).round(2)

### ✍️ Exercise 2: the noise complaint flag

Flag reviews that mention **noise**, **noisy** or **loud** (in any case) and compare their average rating.

First, run the naive version below and look at what it catches.

In [ ]:
naive = reviews["clean"].str.findall(r"(?i)\w*loud\w*").explode().dropna().str.lower()
naive.value_counts().head(8)

😬 Water-filter reviews talk about **cloudy** water. That is a **false positive** for a noise flag.
Fix it with word boundaries `\b` and alternation `(?:a|b|c)`.

In [ ]:
noise_pattern = r"..."   # ✍️ your pattern here
reviews["mentions_noise"] = reviews["clean"].str.contains(noise_pattern, case=False)
reviews.groupby("mentions_noise")["rating"].agg(["count", "mean"]).round(2)

In [ ]:
# ✅ check
n = reviews["mentions_noise"].sum()
assert 300 <= n <= 340, f"{n} flagged: expected about 319. Check your word boundaries."
print(f"✅ {n} noise reviews, rated {reviews.loc[reviews.mentions_noise, 'rating'].mean():.2f} vs "
      f"{reviews.loc[~reviews.mentions_noise, 'rating'].mean():.2f} overall")

## 4 · The manager's question: *how long until it fails?* (10 min)

Customers often write things like *"stopped working after 6 months"*.
If we extract the **number** and the **unit**, we get a rough failure timeline, and nobody had to read 30,000 reviews to get it.

`.str.extract` turns each capturing group into a column:

In [ ]:
after = reviews["clean"].str.extract(r"(?i)\bafter\s+(\d+)\s+(day|week|month|year)s?\b")
after.columns = ["n", "unit"]
after = after.dropna()
print(f"{len(after)} reviews mention 'after <number> <unit>'")
after.head()

In [ ]:
to_months = {"day": 1 / 30, "week": 1 / 4.3, "month": 1, "year": 12}
after["months"] = after["n"].astype(float) * after["unit"].str.lower().map(to_months)
after["rating"] = reviews.loc[after.index, "rating"]

ax = after.query("months <= 36").plot.scatter(x="months", y="rating", alpha=0.4, figsize=(7, 3.5))
ax.set_title("'after N units' mentions: time vs star rating");

### Read before you trust it

Look at some of the matches in context. Does every *"after N months"* describe the **product failing**?

In [ ]:
def in_context(i, pattern=r"(?i)after\s+\d+\s+\w+", width=70):
    text = reviews.at[i, "clean"]
    m = re.search(pattern, text)
    return "…" + text[max(0, m.start() - width): m.end() + width] + "…"

for i in after.sample(6, random_state=1).index:
    print(f"[{reviews.at[i, 'rating']}★]", in_context(i), "\n")

Some matches are about the **old** appliance (*"my washer died after 12 years, this part fixed it"*), and some are *good* news
(*"after 6 months, still works great"*). A regex finds **a pattern**, but it doesn't understand what the sentence means.

### ✍️ Exercise 3 (stretch): number words

People also write *"after **two** weeks"* or *"after **a few** months"*. Extend the pattern to catch
`one`–`six`, `a few` and `several`, and count how many **more** reviews you find.

In [ ]:
wide_pattern = r"..."   # ✍️ your pattern here
wide = reviews["clean"].str.extract(wide_pattern).dropna()
print(len(wide), "vs", len(after))

## Recap

| idea | what you did |
|---|---|
| **Precision** vs **recall** | `loud` also caught *cloudy* (a false positive); the model-number pattern missed `K200` (a false negative) |
| Regex scales | the same pattern ran on 30,000 rows in under a second |
| Regex doesn't understand text | *"after 12 years"* was about the old machine, not this product |

➡️ After the break, you'll use these skills on a **support inbox full of personal data**.